<a href="https://colab.research.google.com/github/avindumihisara0229-code/ErgoSense/blob/Tharusha/DSGP_SVM_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install librosa soundfile

In [ ]:
import os
import numpy as np
import pandas as pd
import librosa

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [9]:
DATASET_PATH = "/content/drive/MyDrive/Colab Notebooks/archive"

In [ ]:
def get_stress_label(filename):
    emotion_code = int(filename.split("-")[2])

    if emotion_code in [5, 6]:
        return 1
    elif emotion_code in [1, 2, 3]:
        return 0
    else:
        return None

In [ ]:
def extract_features(file_path):
    y, sr = librosa.load(file_path, duration=3, offset=0.5)

    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    mfccs_mean = np.mean(mfccs, axis=1)

    pitches, _ = librosa.piptrack(y=y, sr=sr)
    pitch_mean = np.mean(pitches[pitches > 0]) if np.any(pitches > 0) else 0

    energy = np.mean(librosa.feature.rms(y=y))

    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)

    return np.hstack([mfccs_mean, pitch_mean, energy, tempo])

In [10]:
features = []
labels = []

for actor_folder in os.listdir(DATASET_PATH):
    actor_path = os.path.join(DATASET_PATH, actor_folder)

    if os.path.isdir(actor_path):
        for file in os.listdir(actor_path):
            if file.endswith(".wav"):
                label = get_stress_label(file)

                if label is not None:
                    file_path = os.path.join(actor_path, file)
                    feature_vector = extract_features(file_path)

                    features.append(feature_vector)
                    labels.append(label)